# 2. Technical Asset Onboarding (Resource Explorer)

This workbook demonstrates how to use Egeria's template-based **Resource Explorer** view service client (`AutomatedCuration`) to catalog existing data stores (the PostgreSQL `coco_pharma` database and the `uk_sales_forecast.csv` spreadsheet).

In [ ]:
import sys
import os
sys.path.insert(0, '/Users/dwolfson/localGit/egeria-v6/egeria-advisor/data/repos/egeria-python')

from pyegeria import EgeriaTech, settings, config_logging
from pyegeria.omvs.automated_curation import AutomatedCuration

config_logging()
app_config = settings.Environment

EGERIA_USER = 'erinoverview'
EGERIA_USER_PASSWORD = 'secret'

client = EgeriaTech(app_config.egeria_view_server,
                    app_config.egeria_view_server_url,
                    EGERIA_USER, EGERIA_USER_PASSWORD)
token = client.create_egeria_bearer_token(client.user_id, client.user_pwd)

curation_client = AutomatedCuration(app_config.egeria_view_server,
                                    app_config.egeria_view_server_url,
                                    EGERIA_USER, EGERIA_USER_PASSWORD, token)

print("Connected to Resource Explorer.")

## 1. Onboard UK Sales Forecast CSV Spreadsheet

We will onboard the local UK Sales forecast CSV spreadsheet as a `CSVFile` asset in Egeria.

In [ ]:
uk_csv_path = os.path.abspath("uk_sales_forecast.csv")

try:
    uk_guid = curation_client.create_csv_data_file_element_from_template(
        file_name="uk_sales_forecast.csv",
        file_type="CSV Data File",
        file_path_name=uk_csv_path,
        version_identifier="1.0",
        file_encoding="UTF-8",
        file_extension="csv",
        file_system_name="LocalFileSystem",
        description="UK Sales Forecast Spreadsheet containing historical and current pipeline data."
    )
    print(f"UK Sales Forecast CSV onboarded. GUID: {uk_guid}")
except Exception as e:
    print("Error onboarding UK CSV file:", e)

## 2. Onboard PostgreSQL Databases and Register Catalog Targets

We onboard the PostgreSQL database `coco_pharma` running on port 5442 as an asset.

In [ ]:
try:
    # We connect as egeria_user (user4egeria) so that the metadata cataloguer has alignment 
    # of permissions and schema access during survey and onboarding.
    db_guid = curation_client.create_postgres_database_element_from_template(
        postgres_database="coco_pharma",
        server_name="egeria-shared-postgres",
        host_identifier="localhost",
        port="5442",
        db_user="egeria_user",
        db_pwd="user4egeria",
        description="Coco Pharmaceuticals core transactional and operational database."
    )
    print(f"PostgreSQL database coco_pharma onboarded. GUID: {db_guid}")
except Exception as e:
    print("Error onboarding PostgreSQL database:", e)

## 3. Attach Database to Integration Cataloguer

To trigger automatic surveying and cataloging of database tables and schemas, we register the `coco_pharma` database as a catalog target under the PostgreSQL Server Cataloguer.

In [ ]:
# Search for the PostgreSQL Integration Connector GUID
try:
    # We can retrieve integration connector profiles to find the target connector guid
    connectors = client.find_tech_type_elements(search_string="*PostgreSQLServerIntegrationConnector*")
    if connectors and len(connectors) > 0 and not isinstance(connectors, str):
        connector_guid = connectors[0].get('elementHeader', {}).get('guid')
        print(f"Found integration connector: {connectors[0].get('properties', {}).get('displayName')} | GUID: {connector_guid}")
        
        # Add target
        rel_guid = curation_client.add_catalog_target(
            integ_connector_guid=connector_guid,
            metadata_element_guid=db_guid,
            catalog_target_name="Postgres-CocoPharma-Catalog-Target",
            connection_name="Postgres-CocoPharma-Connection"
        )
        print(f"Catalog target registered. Relationship GUID: {rel_guid}")
    else:
        print("PostgreSQL integration connector not found; please ensure it is running in the integration daemon.")
except Exception as e:
    print("Error adding catalog target:", e)